# Building a Simple ETL Pipeline

**MATH 153 · Data Wrangling / Data Engineering**
Howard University · Department of Mathematics
Moussa Doumbia, Ph.D.

---

### Today

1. What a data pipeline is, and its five parts
2. **E**xtract, **T**ransform, **L**oad — one step at a time
3. Putting the three steps inside one function
4. Running it twice without breaking it
5. The two things every pipeline needs

### Our job today

The Bison Café sends us **one sales file per day**. We will turn three of those files into a
clean table and a daily revenue report — and then turn the whole thing into a pipeline we can
run tomorrow without thinking about it.

---
# Part 1. What is a data pipeline?

> A **data pipeline** is a series of steps that moves data from where it *is* to where it is
> *useful*, doing the same thing every time it runs.

The important words are **every time it runs**. A notebook you run once by hand is an analysis.
A pipeline runs tomorrow, and next Tuesday, without you.

### The five components

| # | Component | What it means | In our example |
|---|---|---|---|
| 1 | **Source** | where the data comes from | 3 daily CSV files from the café till |
| 2 | **Extract** | read it in | `pd.read_csv` |
| 3 | **Transform** | make it usable | fix prices, fix names, join the menu |
| 4 | **Load** | put it where people will use it | a SQLite table + a summary CSV |
| 5 | **Schedule** | decide when it runs | every morning at 6 a.m. |

Steps 2, 3 and 4 are so common that they have a name of their own: **ETL**.

```
  SOURCE          EXTRACT        TRANSFORM         LOAD          SCHEDULE
3 CSV files  -->  read them  --> clean & join  --> cafe.db   <-- run daily
  menu.csv                                         summary.csv    at 6 a.m.
```

### ETL in one sentence each

- **Extract** — get the raw data, *without changing it*.
- **Transform** — clean it, reshape it, combine it, calculate from it.
- **Load** — write the result somewhere other people can read.

> You will also hear **ELT**: load the raw data first, transform it later inside the database.
> Same three letters, different order. Today we do ETL.

---
# Part 2. Look at the data first

Never write a pipeline for data you have not looked at.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

DATA = Path("data")
print("files we have:", sorted(p.name for p in DATA.iterdir()))

files we have: ['menu.csv', 'sales_2027-03-02.csv', 'sales_2027-03-03.csv', 'sales_2027-03-04.csv']


In [2]:
peek = pd.read_csv(DATA / "sales_2027-03-02.csv")
print("shape:", peek.shape)
peek.head()

shape: (36, 6)


,sale_id,sale_date,sale_time,item,qty,unit_price
0,S-5001,2027-03-02,14:32,FRUIT CUP,2,$4.00
1,S-5002,2027-03-02,15:22,Turkey Wrap,2,$7.50
2,S-5003,2027-03-02,18:34,Iced Tea,1,$2.95
3,S-5004,2027-03-02,11:58,latte,3,$4.25
4,S-5005,2027-03-02,17:50,latte,1,$4.25


Two things are already wrong, and both are normal.

1. **`unit_price` is text, not a number.** It says `$4.25`. You cannot multiply by `$4.25`.
2. **`item` is spelled inconsistently** — `FRUIT CUP`, `latte`, `  Bagel ` — because the till
   lets staff type it by hand.

Fixing those two things is what "Transform" will mean today.

In [3]:
print("How many different spellings of item names are there?")
print(sorted(peek["item"].unique()))

How many different spellings of item names are there?
['  Drip Coffee ', '  Muffin ', '  Turkey Wrap ', 'BAGEL', 'Bagel', 'Drip Coffee', 'FRUIT CUP', 'Fruit Cup', 'Hot Chocolate', 'ICED TEA', 'Iced Tea', 'LATTE', 'Latte', 'Muffin', 'TURKEY WRAP', 'Turkey Wrap', 'drip coffee', 'fruit cup', 'latte', 'turkey wrap']


In [4]:
menu = pd.read_csv(DATA / "menu.csv")
menu

,item,category,menu_price
0,Latte,Drink,4.25
1,Drip Coffee,Drink,2.50
2,Hot Chocolate,Drink,3.75
3,Iced Tea,Drink,2.95
4,Bagel,Food,2.25
5,Muffin,Food,3.10
6,Turkey Wrap,Food,7.50
7,Fruit Cup,Food,4.00


The menu is our **reference table**: it tells us the category of each item. We will join it
on in the Transform step. Notice it spells every item one correct way — which is exactly why our
messy item names have to be cleaned *before* we try to join.

---
# Part 3. **E** — Extract

Read the raw files. Change nothing. The only job of Extract is to get the data into Python.

In [5]:
def extract(data_dir):
    """Read every daily sales file and stack them into one DataFrame."""
    files = sorted(data_dir.glob("sales_2027-*.csv"))
    frames = [pd.read_csv(f) for f in files]
    return pd.concat(frames, ignore_index=True)

raw = extract(DATA)
print("files read:", 3)
print("rows      :", len(raw))
raw.head(3)

files read: 3
rows      : 107


,sale_id,sale_date,sale_time,item,qty,unit_price
0,S-5001,2027-03-02,14:32,FRUIT CUP,2,$4.00
1,S-5002,2027-03-02,15:22,Turkey Wrap,2,$7.50
2,S-5003,2027-03-02,18:34,Iced Tea,1,$2.95


107 rows from three files. `pd.concat` stacks them; `ignore_index=True` renumbers the rows
0, 1, 2, … so we do not end up with three rows all numbered 0.

**That is all Extract does.** It is deliberately boring. Keeping it boring means that if something
goes wrong later, you know the raw data is untouched and you can start again.

---
# Part 4. **T** — Transform

Four small steps. We will do them one at a time and look after each one.

### Step 1 — turn the price into a number

In [6]:
df = raw.copy()                       # work on a copy, keep `raw` untouched

df["unit_price"] = df["unit_price"].str.replace("$", "", regex=False).astype(float)

print("before:", raw["unit_price"].iloc[0], " (a", type(raw["unit_price"].iloc[0]).__name__ + ")")
print("after :", df["unit_price"].iloc[0], " (a", type(df["unit_price"].iloc[0]).__name__ + ")")

before: $4.00  (a str)
after : 4.0  (a float64)


### Step 2 — turn the date into a real date

In [7]:
df["sale_date"] = pd.to_datetime(df["sale_date"], format="%Y-%m-%d")

print("dtype:", df["sale_date"].dtype)
print("now we can ask for the day of the week:", df["sale_date"].dt.day_name().iloc[0])

dtype: datetime64[us]
now we can ask for the day of the week: Tuesday


A date stored as text is just a string — you cannot sort it reliably, subtract it, or ask what
day of the week it was. Converting it once, here, means every later step gets a real date.

### Step 3 — clean the item names

In [8]:
before = df["item"].nunique()

df["item"] = df["item"].str.strip().str.title()

print(f"distinct spellings before: {before}")
print(f"distinct item names after: {df['item'].nunique()}")
print(sorted(df["item"].unique()))

distinct spellings before: 29
distinct item names after: 8
['Bagel', 'Drip Coffee', 'Fruit Cup', 'Hot Chocolate', 'Iced Tea', 'Latte', 'Muffin', 'Turkey Wrap']


**29 spellings became 8 items.** Two tiny methods did it:

- `.str.strip()` removes spaces at the start and end — `"  Latte "` becomes `"Latte"`
- `.str.title()` capitalises each word — `"LATTE"` and `"latte"` both become `"Latte"`

This step had to come **before** the join. `"latte"` would not have matched `"Latte"` in the menu.

### Step 4 — join the menu, then calculate

In [9]:
df = df.merge(menu, on="item", how="left")

print("rows after the join:", len(df), "(should still be 107)")
print("rows that found no menu match:", df["category"].isna().sum())

rows after the join: 107 (should still be 107)
rows that found no menu match: 0


In [10]:
df["revenue"] = (df["qty"] * df["unit_price"]).round(2)

df[["sale_id", "sale_date", "item", "category", "qty", "unit_price", "revenue"]].head()

,sale_id,sale_date,item,category,qty,unit_price,revenue
0,S-5001,2027-03-02,Fruit Cup,Food,2,4.00,8.00
1,S-5002,2027-03-02,Turkey Wrap,Food,2,7.50,15.00
2,S-5003,2027-03-02,Iced Tea,Drink,1,2.95,2.95
3,S-5004,2027-03-02,Latte,Drink,3,4.25,12.75
4,S-5005,2027-03-02,Latte,Drink,1,4.25,4.25


Two checks worth making every time you join:

- **Did the row count stay the same?** 107 in, 107 out. If it grew, the reference table has
  duplicate rows and every sale got counted twice.
- **Did anything fail to match?** Zero unmatched. If our name cleaning had missed a spelling,
  that item would have a blank category and its revenue would vanish from the report.

### The whole Transform, in one function

In [11]:
def transform(raw, menu):
    """Clean the raw sales and add the columns the report needs."""
    df = raw.copy()
    df["unit_price"] = df["unit_price"].str.replace("$", "", regex=False).astype(float)
    df["sale_date"] = pd.to_datetime(df["sale_date"], format="%Y-%m-%d")
    df["item"] = df["item"].str.strip().str.title()
    df = df.merge(menu, on="item", how="left")
    df["revenue"] = (df["qty"] * df["unit_price"]).round(2)
    return df

clean = transform(raw, menu)
print("clean rows:", len(clean), "| total revenue: $%.2f" % clean["revenue"].sum())

clean rows: 107 | total revenue: $868.35


---
# Part 5. **L** — Load

Where does the result go? Two places, for two different audiences.

- A **database table**, for anyone who wants to query it
- A **summary CSV**, for the café manager who just wants the daily numbers

In [12]:
summary = (clean
           .assign(day=clean["sale_date"].dt.date)
           .groupby(["day", "category"], as_index=False)
           .agg(items_sold=("qty", "sum"), revenue=("revenue", "sum"))
           .rename(columns={"day": "sale_date"}))
summary

,sale_date,category,items_sold,revenue
0,2027-03-02,Drink,35,119.75
1,2027-03-02,Food,40,222.85
2,2027-03-03,Drink,48,162.80
3,2027-03-03,Food,29,112.40
4,2027-03-04,Drink,39,124.30
5,2027-03-04,Food,29,126.25


In [13]:
def load(clean, summary, db_path="cafe.db", csv_path="daily_summary.csv"):
    """Write the results where people will actually read them."""
    out = clean.copy()
    out["sale_date"] = out["sale_date"].dt.strftime("%Y-%m-%d")   # dates as text for SQLite

    con = sqlite3.connect(db_path)
    out.to_sql("sales", con, if_exists="replace", index=False)
    con.close()

    summary.to_csv(csv_path, index=False)
    return len(out)

n = load(clean, summary)
print(f"loaded {n} rows into cafe.db and wrote daily_summary.csv")

loaded 107 rows into cafe.db and wrote daily_summary.csv


In [14]:
# Prove it is really in the database -- ask it a question in SQL.
con = sqlite3.connect("cafe.db")
pd.read_sql_query("""
    SELECT category, ROUND(SUM(revenue), 2) AS revenue
    FROM   sales
    GROUP  BY category
    ORDER  BY revenue DESC
""", con)

,category,revenue
0,Food,461.50
1,Drink,406.85


---
# Part 6. The whole pipeline, in one function

Everything above, in twelve lines. **This is the deliverable.** Not the notebook — the function.

In [15]:
def run_pipeline(data_dir=DATA, db_path="cafe.db", csv_path="daily_summary.csv"):
    """Extract -> Transform -> Load. Run this every morning."""
    menu = pd.read_csv(data_dir / "menu.csv")

    raw = extract(data_dir)                                   # E
    clean = transform(raw, menu)                              # T

    summary = (clean
               .assign(day=clean["sale_date"].dt.date)
               .groupby(["day", "category"], as_index=False)
               .agg(items_sold=("qty", "sum"), revenue=("revenue", "sum"))
               .rename(columns={"day": "sale_date"}))

    load(clean, summary, db_path, csv_path)                   # L
    return clean, summary

In [16]:
clean, summary = run_pipeline()
print("rows:", len(clean), "| revenue: $%.2f" % clean["revenue"].sum())

rows: 107 | revenue: $868.35


Notice what we gained by writing it as a function:

- It **has a name**. You can tell someone "run `run_pipeline()`" instead of "open the notebook and
  run cells 4 through 19 but not cell 12".
- It takes **arguments**. Point it at a different folder and it works on different data.
- It can be **scheduled**. A computer can call a function at 6 a.m.; it cannot click through
  a notebook.

---
# Part 7. Run it twice

A pipeline that breaks when you run it twice is a pipeline nobody can trust. Let's test ours.

In [17]:
run_pipeline()
run_pipeline()          # run it a second time -- on purpose

con = sqlite3.connect("cafe.db")
print("rows in the sales table:", pd.read_sql_query("SELECT COUNT(*) AS n FROM sales", con)["n"][0])
con.close()

rows in the sales table: 107


Still 107, not 214. Running it twice gave the same answer as running it once.

That property has a name: the pipeline is **idempotent**. It is `if_exists="replace"` in our `load`
function that gives it to us — each run throws away the old table and writes a fresh one.

Compare the alternative:

```python
out.to_sql("sales", con, if_exists="append", index=False)   # DON'T
```

With `append`, every run adds another copy. Run it three times and the café has sold everything
three times. **This is the single most common bug in a first pipeline**, and the fix is one
keyword.

---
# Part 8. The two things every pipeline needs

Our pipeline works. It is not yet something you would leave running unattended. Two small
additions change that.

### 1. A log — so you can see what it did

In [18]:
def run_pipeline_v2(data_dir=DATA, db_path="cafe.db", csv_path="daily_summary.csv"):
    menu = pd.read_csv(data_dir / "menu.csv")

    raw = extract(data_dir)
    print(f"  EXTRACT   {len(raw):>4} rows read")

    clean = transform(raw, menu)
    print(f"  TRANSFORM {len(clean):>4} rows cleaned, {clean['item'].nunique()} distinct items")

    summary = (clean
               .assign(day=clean["sale_date"].dt.date)
               .groupby(["day", "category"], as_index=False)
               .agg(items_sold=("qty", "sum"), revenue=("revenue", "sum"))
               .rename(columns={"day": "sale_date"}))

    load(clean, summary, db_path, csv_path)
    print(f"  LOAD      {len(clean):>4} rows written, revenue ${clean['revenue'].sum():.2f}")
    return clean, summary

clean, summary = run_pipeline_v2()

  EXTRACT    107 rows read
  TRANSFORM  107 rows cleaned, 8 distinct items


  LOAD       107 rows written, revenue $868.35


Three print statements. Now when the job runs at 6 a.m. and something looks odd, you can see
**which step** the rows disappeared at — instead of guessing.

### 2. A check — so it refuses to publish nonsense

An **assertion** is a sentence that must be true. If it is not, Python stops immediately and tells
you why. Two are enough here.

In [19]:
# Check 1: the join must not lose any sales
assert clean["category"].notna().all(), "some items did not match the menu"

# Check 2: revenue cannot be negative or zero
assert (clean["revenue"] > 0).all(), "a sale has zero or negative revenue"

print("both checks passed -- safe to publish")

both checks passed -- safe to publish


Put these **between Transform and Load**. That is the right place, because it is the last
moment before the numbers reach somebody else.

Here is what happens when a check fails — this cell is *supposed* to error.

In [20]:
broken = clean.copy()
broken.loc[0, "revenue"] = -5.0          # pretend the transform has a bug

try:
    assert (broken["revenue"] > 0).all(), "a sale has zero or negative revenue"
except AssertionError as e:
    print("Pipeline stopped:", e)

Pipeline stopped: a sale has zero or negative revenue


The pipeline stopped **before** writing to the database. The café manager's report is still
yesterday's — which is correct, and much better than today's report being wrong.

---
# Part 9. The same thing in SQL

Everything we did in pandas can be done in SQL once the data is in the database. Same idea,
different language.

In [21]:
con = sqlite3.connect("cafe.db")

pd.read_sql_query("""
    SELECT sale_date,
           category,
           SUM(qty)              AS items_sold,
           ROUND(SUM(revenue),2) AS revenue
    FROM   sales
    GROUP  BY sale_date, category
    ORDER  BY sale_date, category
""", con)

,sale_date,category,items_sold,revenue
0,2027-03-02,Drink,35,119.75
1,2027-03-02,Food,40,222.85
2,2027-03-03,Drink,48,162.80
3,2027-03-03,Food,29,112.40
4,2027-03-04,Drink,39,124.30
5,2027-03-04,Food,29,126.25


In [22]:
# Which items earn the most?
pd.read_sql_query("""
    SELECT item, SUM(qty) AS sold, ROUND(SUM(revenue),2) AS revenue
    FROM   sales
    GROUP  BY item
    ORDER  BY revenue DESC
    LIMIT  5
""", con)

,item,sold,revenue
0,Turkey Wrap,36,270.00
1,Latte,32,136.00
2,Iced Tea,38,112.10
3,Hot Chocolate,23,86.25
4,Fruit Cup,20,80.00


In [23]:
con.close()
print("done")

done


**pandas or SQL?** A rough rule:

| Use **pandas** when | Use **SQL** when |
|---|---|
| reading files and cleaning text | the data is already in a database |
| the step is awkward to express in SQL | the data is too big for memory |
| you are still exploring | the question is a standard group-and-total |

Most real pipelines use both: pandas to clean and load, SQL to query afterwards. That is exactly
what we did.

---
# Summary

### The five components
**Source → Extract → Transform → Load**, on a **Schedule**.

### What we built
```python
def run_pipeline(data_dir):
    menu  = pd.read_csv(...)      # reference data
    raw   = extract(data_dir)     # E -- read, change nothing
    clean = transform(raw, menu)  # T -- fix types, fix text, join, calculate
    assert ...                    #      check before you publish
    load(clean, summary)          # L -- write where people will read it
```

### Five things to remember

1. **Extract changes nothing.** Keep the raw data raw, so you can always start over.
2. **Clean before you join.** `"latte"` does not match `"Latte"`.
3. **Check the row count after every join.** Same number in, same number out.
4. **Make it runnable twice.** `if_exists="replace"`, not `"append"`.
5. **Put the checks between Transform and Load** — the last moment before the numbers reach
   somebody else.

### The one-sentence version

> A pipeline is just **three functions and a schedule** — and the difference between a pipeline
> and a notebook is that somebody else can run it.